# Wildlife Tracker - Exploratory Data Analysis

This notebook explores the collected and processed wildlife data:
1. Species count by taxonomic group
2. Geographic coverage across Indian states
3. Conservation status distribution
4. Chunk length distribution
5. Sample chunks inspection

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load data
PROCESSED_DIR = Path("../data/processed")
CHUNKS_DIR = Path("../data/chunks")

with open(PROCESSED_DIR / "all_species.json", encoding="utf-8") as f:
    all_species = json.load(f)

with open(CHUNKS_DIR / "all_chunks.json", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"Total species: {len(all_species)}")
print(f"Total chunks: {len(all_chunks)}")

## 1. Species Count by Taxonomic Group

In [ ]:
groups = Counter(sp["taxonomic_group"] for sp in all_species)
group_df = pd.DataFrame(groups.items(), columns=["Group", "Count"]).sort_values("Count", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = {"birds": "#4CAF50", "mammals": "#FF9800", "reptiles": "#2196F3"}
bars = ax.bar(group_df["Group"], group_df["Count"], 
              color=[colors.get(g, "#9E9E9E") for g in group_df["Group"]])
ax.set_title("Species Count by Taxonomic Group", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Species")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            str(int(bar.get_height())), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nBreakdown:")
for _, row in group_df.iterrows():
    print(f"  {row['Group']}: {row['Count']} species")

## 2. Geographic Coverage - Indian States

In [ ]:
state_counts = Counter()
for sp in all_species:
    for state in sp.get("geographic_regions", []):
        state_counts[state] += 1

state_df = pd.DataFrame(state_counts.items(), columns=["State", "Species Count"])
state_df = state_df.sort_values("Species Count", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(8, len(state_df) * 0.35)))
ax.barh(state_df["State"], state_df["Species Count"], color="#4CAF50", alpha=0.8)
ax.set_title("Species Records by Indian State", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Species")
plt.tight_layout()
plt.show()

print(f"\nStates with most species: {state_df.tail(5)[['State','Species Count']].to_string(index=False)}")
print(f"States with fewest species: {state_df.head(5)[['State','Species Count']].to_string(index=False)}")
print(f"\nSpecies with no region data: {sum(1 for sp in all_species if not sp.get('geographic_regions'))}")

## 3. Conservation Status Distribution

In [ ]:
status_counts = Counter(sp.get("conservation_status", "Not Evaluated") or "Not Evaluated" for sp in all_species)
status_order = ["Least Concern", "Near Threatened", "Vulnerable", "Endangered", 
                "Critically Endangered", "Data Deficient", "Not Evaluated"]
status_colors = {
    "Least Concern": "#4CAF50",
    "Near Threatened": "#8BC34A", 
    "Vulnerable": "#FF9800",
    "Endangered": "#FF5722",
    "Critically Endangered": "#F44336",
    "Data Deficient": "#9E9E9E",
    "Not Evaluated": "#BDBDBD",
}

ordered = [(s, status_counts.get(s, 0)) for s in status_order if status_counts.get(s, 0) > 0]
labels, values = zip(*ordered) if ordered else ([], [])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
bar_colors = [status_colors.get(l, "#9E9E9E") for l in labels]
ax1.bar(range(len(labels)), values, color=bar_colors)
ax1.set_xticks(range(len(labels)))
ax1.set_xticklabels(labels, rotation=45, ha="right")
ax1.set_title("Conservation Status Distribution", fontweight="bold")
ax1.set_ylabel("Number of Species")

# Pie chart
ax2.pie(values, labels=labels, colors=bar_colors, autopct="%1.1f%%", startangle=90)
ax2.set_title("Conservation Status Proportions", fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Chunk Length Distribution

In [ ]:
token_counts = [c["token_estimate"] for c in all_chunks]
char_counts = [len(c["text"]) for c in all_chunks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(token_counts, bins=30, color="#4CAF50", alpha=0.8, edgecolor="white")
ax1.axvline(x=500, color="red", linestyle="--", label=f"Target: 500 tokens")
ax1.set_title("Chunk Size (Estimated Tokens)", fontweight="bold")
ax1.set_xlabel("Tokens")
ax1.set_ylabel("Count")
ax1.legend()

ax2.hist(char_counts, bins=30, color="#2196F3", alpha=0.8, edgecolor="white")
ax2.set_title("Chunk Size (Characters)", fontweight="bold")
ax2.set_xlabel("Characters")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Token stats: min={min(token_counts)}, max={max(token_counts)}, "
      f"mean={sum(token_counts)/len(token_counts):.0f}, median={sorted(token_counts)[len(token_counts)//2]}")

# By section type
section_counts = Counter(c["section_type"] for c in all_chunks)
print(f"\nChunks by section type:")
for sec, count in section_counts.most_common():
    print(f"  {sec}: {count}")

## 5. Sample Chunks Inspection

In [ ]:
# Show 5 sample chunks from different species/sections
import random
random.seed(42)

samples = random.sample(all_chunks, min(5, len(all_chunks)))

for i, chunk in enumerate(samples):
    print(f"{'='*80}")
    print(f"SAMPLE CHUNK {i+1}")
    print(f"{'='*80}")
    print(f"Chunk ID:     {chunk['chunk_id']}")
    print(f"Species:      {chunk['species_name']} ({chunk['scientific_name']})")
    print(f"Section:      {chunk['section_type']}")
    print(f"Group:        {chunk['taxonomic_group']}")
    print(f"Status:       {chunk['conservation_status']}")
    print(f"Regions:      {', '.join(chunk['geographic_regions'][:5]) if chunk['geographic_regions'] else 'N/A'}")
    print(f"Tokens (est): {chunk['token_estimate']}")
    print(f"\nTEXT:")
    print(chunk['text'][:500])
    if len(chunk['text']) > 500:
        print(f"... [{len(chunk['text']) - 500} more chars]")
    print()

## 6. Data Quality Summary

In [ ]:
# Overall quality metrics
total_sp = len(all_species)
with_intro = sum(1 for sp in all_species if sp["description"].get("introduction"))
with_habitat = sum(1 for sp in all_species if sp["description"].get("habitat"))
with_phys = sum(1 for sp in all_species if sp["description"].get("physical_description"))
with_behavior = sum(1 for sp in all_species if sp["description"].get("behavior"))
with_regions = sum(1 for sp in all_species if sp.get("geographic_regions"))
with_conservation = sum(1 for sp in all_species if sp.get("conservation_status") and sp["conservation_status"] != "Not Evaluated")

quality = pd.DataFrame({
    "Metric": ["Has Introduction", "Has Habitat Info", "Has Physical Description", 
               "Has Behavior Info", "Has Region Data", "Has Conservation Status"],
    "Count": [with_intro, with_habitat, with_phys, with_behavior, with_regions, with_conservation],
    "Percentage": [f"{x/total_sp*100:.1f}%" for x in [with_intro, with_habitat, with_phys, with_behavior, with_regions, with_conservation]]
})

print(f"Data Quality Report ({total_sp} species)")
print("=" * 50)
print(quality.to_string(index=False))
print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Avg chunks per species: {len(all_chunks)/total_sp:.1f}")